In [1]:
from __future__ import annotations
import json, random
from collections import defaultdict
from pathlib import Path

In [ ]:
# tools/split_manifest.py
from __future__ import annotations
import json, random
from collections import defaultdict
from pathlib import Path

MANIFEST = Path("/Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/ambulance_dataset_fast_150_espisode_cpu_30_senario/manifests/episodes.jsonl")
OUT_DIR  = MANIFEST.parent
RATIOS   = (0.8, 0.1, 0.1)  # train, val, test
SEED     = 42

def load_manifest(p: Path):
    items = []
    with p.open() as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                items.append(json.loads(ln))
    return items

def stratified_split(items, ratios):
    random.seed(SEED)
    by_scen = defaultdict(list)
    for it in items: by_scen[it["scenario"]].append(it)

    train, val, test = [], [], []
    for scen, arr in by_scen.items():
        random.shuffle(arr)
        n = len(arr)
        n_tr = int(round(ratios[0]*n))
        n_va = int(round(ratios[1]*n))
        # put the remainder into test to keep totals consistent
        n_te = n - n_tr - n_va
        train += arr[:n_tr]
        val   += arr[n_tr:n_tr+n_va]
        test  += arr[n_tr+n_va:]
    return train, val, test

def write_jsonl(path: Path, items):
    with path.open("w") as f:
        for it in items: f.write(json.dumps(it) + "\n")

if __name__ == "__main__":
    items = load_manifest(MANIFEST)
    tr, va, te = stratified_split(items, RATIOS)
    write_jsonl(OUT_DIR/"episodes_train.jsonl", tr)
    write_jsonl(OUT_DIR/"episodes_val.jsonl",   va)
    write_jsonl(OUT_DIR/"episodes_test.jsonl",  te)
    print("[ok]", len(tr), len(va), len(te))
